In [1]:
    import warnings
    warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading private GCS files
customers = pd.read_csv("gs://ppnj_data/files/dim_customer.csv")
travel_package = pd.read_csv("gs://ppnj_data/files/dim_travel_package.csv")
sales = pd.read_csv("gs://ppnj_data/files/fact_sales.csv")
travel_guide = pd.read_csv("gs://ppnj_data/files/dim_travel_guide.csv")
feedback = pd.read_csv("gs://ppnj_data/files/dim_feedback.csv")
refund = pd.read_csv("gs://ppnj_data/files/dim_refund.csv")


## What if Refund Rate Drops by 20%?

In [3]:
'''
Simulation Steps:
1. Calculate current revenue loss due to refunds
2. Reduce refund amount by 20%
3. Show impact on net revenue
'''

# 1. Merge refund with sales and travel_package
merged_refund = refund.merge(sales[['booking_id', 'package_id', 'num_of_pax']], on='booking_id')
merged_refund = merged_refund.merge(travel_package[['package_id', 'price_per_pax']], on='package_id')

# 2. Calculate original total value of each booking
merged_refund['trip_value'] = merged_refund['num_of_pax'] * merged_refund['price_per_pax']

# 3. Simulate reduced refund scenario
merged_refund['reduced_refund'] = merged_refund['refund_amount'] * 0.8
original_total_refund = merged_refund['refund_amount'].sum()
new_total_refund = merged_refund['reduced_refund'].sum()

impact = original_total_refund - new_total_refund

print(f"Original Refund Total: RM{original_total_refund:,.2f}")
print(f"After 20% Reduction: RM{new_total_refund:,.2f}")
print(f"Revenue Retained: RM{impact:,.2f}")


Original Refund Total: RM49,115.00
After 20% Reduction: RM39,292.00
Revenue Retained: RM9,823.00


## What if 10% of solo travelers upgrade to family packages?

In [4]:
'''
Simulation Steps:
1. Find avg revenue for solo vs. family
2. Estimate how many solo bookings could shift
3. Multiply by price difference × avg pax
'''

# Merge to get package types and prices
merged_sales = sales.merge(travel_package[['package_id', 'package_type', 'price_per_pax']], on='package_id')

# Filter solo and family bookings
solo = merged_sales[merged_sales['package_type'] == 'solo']
family = merged_sales[merged_sales['package_type'] == 'family']

# Calculate average revenue per booking (price × pax)
solo['booking_value'] = solo['price_per_pax'] * solo['num_of_pax']
family['booking_value'] = family['price_per_pax'] * family['num_of_pax']

solo_avg = solo['booking_value'].mean()
family_avg = family['booking_value'].mean()

# Simulate: 10% of solo customers upgrade
upgrade_count = int(0.10 * len(solo))
extra_revenue = upgrade_count * (family_avg - solo_avg)

print(f"Avg Solo Booking Value: RM{solo_avg:,.2f}")
print(f"Avg Family Booking Value: RM{family_avg:,.2f}")
print(f"Potential Upgrades: {upgrade_count} bookings")
print(f"Extra Revenue from Upgrade Campaign: RM{extra_revenue:,.2f}")


Avg Solo Booking Value: RM1,941.68
Avg Family Booking Value: RM6,181.84
Potential Upgrades: 12 bookings
Extra Revenue from Upgrade Campaign: RM50,881.84


## Shift campaigns to peak months (July–August)

| Row | Month | Month Name | Total Bookings | Total Gross Revenue | Month Rank |
|-----|-------|-------------|----------------|----------------------|-------------|
| 1   | 8     | August      | 138            | 1,155,382            | 1           |
| 2   | 7     | July        | 147            | 1,063,219            | 3           |
| 3   | 6     | June        | 124            | 1,068,295            | 2           |


In [5]:
'''
Assumptions:
1.  & August have historically higher booking volume
2. We simulate a 20% campaign push — increasing bookings in those months
'''

# Extract month of travel
sales['travel_month'] = pd.to_datetime(sales['travel_date']).dt.month_name()

# Get average revenue per booking
sales['booking_value'] = sales['num_of_pax'] * sales.merge(travel_package[['package_id', 'price_per_pax']], on='package_id')['price_per_pax']

# Monthly totals
monthly_revenue = sales.groupby('travel_month')['booking_value'].sum().sort_values(ascending=False)
print(monthly_revenue)

# Simulate 20% lift in July & August
june_august_total = monthly_revenue.get('June', 0) + monthly_revenue.get('July', 0) + monthly_revenue.get('August', 0)
additional_revenue = 0.20 * june_august_total

print(f"June–August baseline: RM{june_august_total:,.2f}")
print(f"Expected revenue gain with campaign: RM{additional_revenue:,.2f}")


travel_month
September    1207447
August        964677
June          948175
July          923932
October       906737
December      879545
April         858087
January       802212
November      792648
May           784718
March         754583
February      601482
Name: booking_value, dtype: int64
June–August baseline: RM2,836,784.00
Expected revenue gain with campaign: RM567,356.80
